# 01 · Deduplication and Identity Resolution

**First notebook in the pipeline. Nothing runs before this.**

## What this does
Assigns every image a photograph id and a patient id, and measures how much the
two corpora overlap. It deletes nothing.

## Why it runs first
Identity is a property of the data, not of the modelling. In the first version of
this project deduplication ran *after* labels and splits already existed, and an
85% leakage rate went unnoticed for months. Resolving identity before anything
else touches the data is what prevents that.

## Why not just group by filename
Inside one Roboflow export, augmented copies share a source id, so filename
grouping works. Across two different exports the same photograph is renumbered, so
filenames report zero overlap even when the corpora share images. We hash the
pixels instead.

## The signature
Each image becomes a 256-bit signature from a 16x16 greyscale grid thresholded at
its median, reduced to the minimum over the 8 dihedral transforms so rotated and
mirrored copies collide. The grid is square on purpose: a non-square grid samples
differently after a 90-degree rotation and the invariance breaks. Signatures are
compared by Hamming distance, not equality, because JPEG re-encoding flips bits
near the threshold.

## Outputs
- `identity_roboflow.csv` — path, photo_id, patient_id, hash_cluster
- `identity_dfuc.csv` — same, for DFUC 2024
- `overlap_report.txt` — shared photographs between the corpora


In [ ]:
# Cell 1 · config and input checks
from pathlib import Path

# ── set these to your local paths ────────────────────────────────────
ROBOFLOW_DIR = Path('data/raw/roboflow_dfu')
DFUC_DIR     = Path('data/raw/dfuc_2024')
OUT_DIR      = Path('data/interim'); OUT_DIR.mkdir(parents=True, exist_ok=True)
# ─────────────────────────────────────────────────────────────────────

GRID = 16          # 16x16 square grid -> 256-bit signature
SEED = 42

# assert inputs exist before doing any work — the ground rule
problems = [f'{n} folder not found: {d}'
            for n, d in [('Roboflow', ROBOFLOW_DIR), ('DFUC 2024', DFUC_DIR)]
            if not d.exists()]
if problems:
    print('STOPPING. Fix these paths and rerun:')
    for p in problems: print('  -', p)
    raise SystemExit(1)
print('input folders found. proceeding.')

In [ ]:
# Cell 2 · the hashing engine
import numpy as np
from PIL import Image

IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

def grid_bits(img):
    g = img.convert('L').resize((GRID, GRID), Image.Resampling.LANCZOS)
    a = np.asarray(g, dtype=np.float32)
    return a > np.median(a)

def canonical_bits(img):
    # 256-bit signature minimised over the 8 dihedral transforms, so a
    # rotated or mirrored copy hashes identically. The grid is SQUARE on
    # purpose: a non-square grid (an ordinary dhash) samples differently
    # after a 90-degree rotation and the invariance breaks.
    m = grid_bits(img)
    best = None
    for mirror in (False, True):
        base = np.fliplr(m) if mirror else m
        for k in range(4):
            v = tuple(np.rot90(base, k).flatten().astype(np.uint8))
            if best is None or v < best:
                best = v
    return np.array(best, dtype=bool)

def pack(bitrows):
    # (N,256) bool -> (N,4) uint64 for fast XOR + popcount
    n, b = bitrows.shape
    words = b // 64
    out = np.zeros((n, words), dtype=np.uint64)
    for w in range(words):
        seg = bitrows[:, w*64:(w+1)*64].astype(np.uint64)
        acc = np.zeros(n, dtype=np.uint64)
        for i in range(64):
            acc = (acc << np.uint64(1)) | seg[:, i]
        out[:, w] = acc
    return out

_POP = np.array([bin(i).count('1') for i in range(256)], dtype=np.uint8)
def hamming(packed, row):
    return _POP[(packed ^ row).view(np.uint8)].sum(axis=1)

print('hashing engine ready')

In [ ]:
# Cell 3 · recover photograph and patient id from a filename
import re

RF_SUFFIX  = re.compile(r'_jpg\.rf\.[0-9A-Za-z]+$')   # Roboflow export tag
AUG_SUFFIX = re.compile(r'(_M|_R\d+)$')               # DFUC mirror / rotation

def photo_id_from_name(path):
    core = RF_SUFFIX.sub('', Path(str(path)).stem)
    prev = None
    while core != prev:
        prev = core
        core = AUG_SUFFIX.sub('', core)
    return core

def patient_id_from_photo(photo_id):
    return str(photo_id).split('_')[0]    # 001630_20 -> 001630

assert photo_id_from_name('001630_20_M_jpg.rf.abc.jpg') == '001630_20'
assert patient_id_from_photo('001630_20') == '001630'
print('filename patterns verified')

In [ ]:
# Cell 4 · build an identity table for one corpus
import pandas as pd

class DSU:
    # accepts either an int count or a list of keys
    def __init__(s, n):
        s.p = {k: k for k in (range(n) if isinstance(n, int) else n)}
    def find(s, x):
        while s.p[x] != x:
            s.p[x] = s.p[s.p[x]]; x = s.p[x]
        return x
    def union(s, a, b):
        ra, rb = s.find(a), s.find(b)
        if ra != rb: s.p[rb] = ra

def hash_corpus(img_dir, label=''):
    paths = sorted(p for p in img_dir.rglob('*')
                   if p.suffix.lower() in IMG_EXT)
    print(f'  {label}: {len(paths):,} image files')
    if not paths:
        raise FileNotFoundError(f'no images under {img_dir}')
    bits = np.zeros((len(paths), GRID*GRID), dtype=bool)
    for i, p in enumerate(paths):
        try:
            with Image.open(p) as im:
                bits[i] = canonical_bits(im)
        except Exception as e:
            print(f'    skip {p.name}: {e}')
        if i and i % 2000 == 0:
            print(f'    hashed {i:,}/{len(paths):,}')
    return paths, pack(bits)

def cluster(packed, radius):
    dsu = DSU(len(packed))
    for i in range(len(packed)):
        d = hamming(packed, packed[i])
        for j in np.where(d <= radius)[0]:
            if j > i: dsu.union(i, j)
    raw = [dsu.find(i) for i in range(len(packed))]
    remap = {c: k for k, c in enumerate(sorted(set(raw)))}
    return [remap[c] for c in raw]

def identity_table(paths, clusters):
    df = pd.DataFrame({
        'path': [str(p) for p in paths],
        'photo_id': [photo_id_from_name(p) for p in paths],
        'hash_cluster': clusters})
    df['patient_id'] = df['photo_id'].map(patient_id_from_photo)
    # photograph identity = union of BOTH signals. Two images are the same
    # photograph if they share a filename photo_id OR a content cluster.
    # The hash can over-split one photo into several clusters when its
    # augmented copies differ at 16x16; the filename catches those. The
    # hash catches duplicates the filename misses. Using both is strictly
    # safer than either alone.
    d = DSU(list(df.index))
    for col in ['photo_id', 'hash_cluster']:
        for _, g in df.groupby(col):
            rows = g.index.tolist()
            for j in rows[1:]:
                d.union(rows[0], j)
    roots = {r: k for k, r in enumerate(sorted({d.find(i) for i in df.index}))}
    df['photo_unit'] = [roots[d.find(i)] for i in df.index]
    return df

print('corpus functions ready')

In [ ]:
# Cell 5 · calibrate the Hamming radius against filenames
# Filenames give ground-truth grouping WITHIN a corpus, so we pick the
# radius that best reproduces filename grouping on Roboflow, then reuse it
# where filenames are useless (across corpora).
from sklearn.metrics import adjusted_rand_score

rf_paths, rf_packed = hash_corpus(ROBOFLOW_DIR, 'Roboflow')
true_groups = [photo_id_from_name(p) for p in rf_paths]

best_t, best_score = 10, -1.0
print('\n  radius : agreement with filename grouping')
for t in [4, 6, 8, 10, 12, 14, 16, 20]:
    score = adjusted_rand_score(true_groups, cluster(rf_packed, t))
    print(f'    {t:2d}   : {score:.4f}')
    if score > best_score:
        best_score, best_t = score, t

HAMMING_NEAR = best_t
print(f'\nchosen radius: {HAMMING_NEAR} (agreement {best_score:.4f})')

In [ ]:
# Cell 6 · build and save both identity tables
rf_clusters = cluster(rf_packed, HAMMING_NEAR)
rf_df = identity_table(rf_paths, rf_clusters)
rf_df.to_csv(OUT_DIR / 'identity_roboflow.csv', index=False)
print(f'wrote {(OUT_DIR / "identity_roboflow.csv").resolve()}')
print(f'  {len(rf_df):,} images, {rf_df.photo_unit.nunique():,} photographs '
      f'(hash alone said {rf_df.hash_cluster.nunique():,}, filename recovered the rest), '
      f'{rf_df.patient_id.nunique():,} patients')

dfuc_paths, dfuc_packed = hash_corpus(DFUC_DIR, 'DFUC 2024')
dfuc_clusters = cluster(dfuc_packed, HAMMING_NEAR)
dfuc_df = identity_table(dfuc_paths, dfuc_clusters)
dfuc_df.to_csv(OUT_DIR / 'identity_dfuc.csv', index=False)
print(f'wrote {(OUT_DIR / "identity_dfuc.csv").resolve()}')
print(f'  {len(dfuc_df):,} images, {dfuc_df.hash_cluster.nunique():,} photographs, '
      f'{dfuc_df.patient_id.nunique():,} patients')

In [ ]:
# Cell 7 · cross-corpus overlap — the independence test
# DFUC is the control experiment. If it shares photographs with Roboflow,
# the control is contaminated. This checks pixel content, not filenames.
def reps(df, packed):
    idx = df.groupby('hash_cluster').head(1).index.values
    return packed[idx]

rf_rep, df_rep = reps(rf_df, rf_packed), reps(dfuc_df, dfuc_packed)
shared = sum((hamming(df_rep, rf_rep[i]) <= HAMMING_NEAR).any()
             for i in range(len(rf_rep)))

pct_rf = shared / len(rf_rep) * 100
pct_df = shared / len(df_rep) * 100
report = (f'Cross-corpus overlap\n'
          f'  Roboflow photographs : {len(rf_rep):,}\n'
          f'  DFUC photographs     : {len(df_rep):,}\n'
          f'  shared               : {shared:,}\n'
          f'  = {pct_rf:.1f}% of Roboflow, {pct_df:.1f}% of DFUC\n')
print(report)
(OUT_DIR / 'overlap_report.txt').write_text(report)
print('VERDICT:', 'independent, control is clean.'
      if max(pct_rf, pct_df) < 2 else 'non-trivial overlap, investigate.')

In [ ]:
# Cell 8 · summary
print('=' * 58)
print('STAGE 01 COMPLETE')
print('=' * 58)
print(f'  identity_roboflow.csv : {len(rf_df):,} images, '
      f'{rf_df.hash_cluster.nunique():,} photographs')
print(f'  identity_dfuc.csv     : {len(dfuc_df):,} images, '
      f'{dfuc_df.hash_cluster.nunique():,} photographs')
print(f'  Hamming radius        : {HAMMING_NEAR}')
print(f'  cross-corpus overlap  : {shared:,} photographs')
print('\nnext: 02_label_derivation.ipynb')